## References:


# Install the latest version of scikit-learn library

In [ ]:
! pip install scikit-learn==1.4

# Install array-api-compat library that supports Array API

In [ ]:
!python -m pip install array-api-compat

# Imports

In [ ]:
import sklearn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.datasets import make_classification
import torch

print(sklearn.__version__)

# Create Classification Data

In [ ]:
X_np, y_np = make_classification(random_state=0, n_samples=500_000, n_features=300)
X_np, y_np

# Globally set configuration

In [ ]:
sklearn.set_config(array_api_dispatch=True)

# LinearDiscriminantAnalysis

### Sklearn Performance on CPU

In [ ]:
import time

t1 = time.perf_counter()

X_torch_cpu, y_torch_cpu = torch.asarray(X_np), torch.asarray(y_np)

lda_torch_cpu = LinearDiscriminantAnalysis()
lda_torch_cpu.fit(X_torch_cpu, y_torch_cpu)

predictions = lda_torch_cpu.predict(X_torch_cpu)

print(predictions.device)
print(type(predictions))

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

### Sklearn Performance on GPU

In [ ]:
X_torch_cuda = torch.asarray(X_np, device="cuda", dtype=torch.float32)
y_torch_cuda = torch.asarray(y_np, device="cuda", dtype=torch.float32)
print(type(X_torch_cuda))
print(type(y_torch_cuda))

In [ ]:
import time

t1 = time.perf_counter()

lda_torch_cuda = LinearDiscriminantAnalysis()
lda_torch_cuda.fit(X_torch_cuda, y_torch_cuda)
predictions = lda_torch_cuda.predict(X_torch_cuda)

print(predictions.device)
print(type(predictions))

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

In [ ]:
print(type(predictions))

# Train model on GPU and transfer to CPU for deployment

In [ ]:
from sklearn.utils._array_api import _estimator_with_converted_arrays

tensor_to_ndarray = lambda array : array.cpu().numpy()
lda_np = _estimator_with_converted_arrays(lda_torch_cuda, tensor_to_ndarray)
X_trans = lda_np.transform(X_np)

print(type(X_np))
print(type(X_trans))

# PCA Decomposition

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

X_np = np.random.rand(1000000, 100)


### Sklearn Performance on CPU

In [ ]:
X_torch_cpu = torch.asarray(X_np)

t1 = time.perf_counter()

pca_cpu = PCA(n_components=2, svd_solver='full')
pca_cpu.fit(X_torch_cpu)
print(pca_cpu.explained_variance_ratio_)
print(pca_cpu.singular_values_)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

### Sklearn Performance on GPU

In [ ]:
X_torch_cuda = torch.asarray(X_np, device="cuda", dtype=torch.float32)

t1 = time.perf_counter()

pca_cuda = PCA(n_components=2, svd_solver='full')
pca_cuda.fit(X_torch_cuda)
print(pca_cuda.explained_variance_ratio_)
print(pca_cuda.singular_values_)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)